# Exploring the raw feed — column by column, then payload by payload

**Goal**: build a mental model of the two parquet files before deciding anything.

**Plan**:
1. Land in the data — dimensions, first rows.
2. Walk the outer (delivery-metadata) columns one group at a time. What does each mean? How does it vary?
3. Open the `payload_json` string — it's the real Open Finance API response. Explore per `(investment_type, payload_kind)` shape.
4. Same for `transaction_json`.
5. End with a scratch-pad of observations to feed into the design docs.

**Spec reference** (keep open in another tab):
- Portal: https://openfinancebrasil.atlassian.net/wiki/spaces/OF/overview
- Investments Swagger UI: https://openbanking-brasil.github.io/openapi/swagger-apis/investments/?urls.primaryName=1.0.1
- Raw YAMLs on GitHub: https://github.com/OpenBanking-Brasil/openapi/tree/main/swagger-apis

## 1 · Landing

In [ ]:
import duckdb, json, pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 40)

DATA = Path('../data/raw')
con = duckdb.connect()
con.execute(f"CREATE VIEW pos AS SELECT * FROM read_parquet('{DATA/'raw_positions.parquet'}')")
con.execute(f"CREATE VIEW txn AS SELECT * FROM read_parquet('{DATA/'raw_transactions.parquet'}')")

print(f'positions:    {con.execute("SELECT count(*) FROM pos").fetchone()[0]:>8,} rows')
print(f'transactions: {con.execute("SELECT count(*) FROM txn").fetchone()[0]:>8,} rows')

### The two schemas

Everything is `VARCHAR` — even timestamps and identifiers. Typing is deferred to the staging layer.

Notice: positions has `payload_kind` and `payload_json`; transactions has `payload_source`, `page`, `query_window`, and `transaction_json`.

In [ ]:
print('--- POSITIONS ---'); print(con.execute('DESCRIBE pos').df().to_string(index=False))
print('\n--- TRANSACTIONS ---'); print(con.execute('DESCRIBE txn').df().to_string(index=False))

### Eyeball a few raw rows

The JSON columns will look like giant strings. That's fine — we'll open them below.

In [ ]:
con.execute('SELECT * FROM pos LIMIT 3').df()

In [ ]:
con.execute('SELECT * FROM txn LIMIT 3').df()

## 2 · Outer columns — delivery metadata

The 15 outer columns are metadata that Decade's ingestion layer wraps around the raw provider payloads. Understanding them tells us *how* records arrive, before we care what's inside them.

We'll walk four groups:
- **Who** — the institution and the customer
- **Where** — account, connection
- **When** — snapshot & ingestion timestamps
- **What** — investment identifiers + which endpoint the payload came from

### 2.1 · Who — `institution_id`, `institution_name`, `party_id`

- **`institution_id`** — Decade's identifier for the source bank. Should map 1:1 with `institution_name`.
- **`institution_name`** — human-readable. Not authoritative; can drift.
- **`party_id`** — the customer. "Party" is Open Finance vocabulary (`PF` = natural person, `PJ` = legal entity).

Watch for: multiple `institution_name`s per `institution_id` (rebrand, typo), or vice-versa (bad joins upstream).

In [ ]:
# Institutions and their volume
con.execute("""
  SELECT institution_id, institution_name,
         count(*) AS positions_rows,
         count(DISTINCT party_id) AS customers
  FROM pos
  GROUP BY 1,2
  ORDER BY positions_rows DESC
""").df()

In [ ]:
# Are institution_id and institution_name in bijection? (they should be)
con.execute("""
  SELECT institution_id, count(DISTINCT institution_name) AS n_names
  FROM pos GROUP BY 1 HAVING count(DISTINCT institution_name) > 1
""").df()

In [ ]:
# Customer scale — 'a few hundred synthetic customers'
con.execute("SELECT count(DISTINCT party_id) AS customers FROM pos").df()

### 2.2 · Where — `account_id`, `connection_id`

- **`account_id`** — the investment account within the institution. A customer can hold accounts at multiple institutions and multiple accounts at one institution.
- **`connection_id`** — Decade's identifier for the customer's *authorized connection* to that institution (an Open Finance consent). One consent can produce many snapshots over time.

Cardinality expectation: `customers ≤ connections ≤ accounts`.

In [ ]:
con.execute("""
  SELECT
    count(DISTINCT party_id)                  AS customers,
    count(DISTINCT connection_id)             AS connections,
    count(DISTINCT account_id)                AS accounts,
    count(DISTINCT (party_id, institution_id)) AS customer_institution_pairs
  FROM pos
""").df()

In [ ]:
# How many accounts does an average customer have?
con.execute("""
  SELECT accounts_per_customer, count(*) AS n_customers
  FROM (SELECT party_id, count(DISTINCT account_id) AS accounts_per_customer FROM pos GROUP BY 1)
  GROUP BY 1 ORDER BY 1
""").df()

### 2.3 · When — `snapshot_created_at`, `ingested_at`, `query_window` (txn only)

- **`snapshot_created_at`** — when the provider's *snapshot* was constructed. The natural per-snapshot event time.
- **`snapshot_id`** — groups all records that came together in one sync from one institution.
- **`ingested_at`** — when Decade's pipeline received/wrote the record. **This is the natural watermark for incremental jobs** (`arrival_time` in case-study parlance).
- **`query_window`** (transactions only) — the date range the provider was asked for. Transactions endpoints in Open Finance are queried with a `fromTransactionDate` / `toTransactionDate`; this records what we asked.

In [ ]:
con.execute("""
  SELECT
    min(snapshot_created_at) AS snapshot_earliest,
    max(snapshot_created_at) AS snapshot_latest,
    min(ingested_at)         AS ingested_earliest,
    max(ingested_at)         AS ingested_latest
  FROM pos
""").df()

In [ ]:
# Snapshot cadence per institution — the case study says 'irregular and differs by institution'
con.execute("""
  SELECT institution_name, count(DISTINCT snapshot_id) AS snapshots,
         min(snapshot_created_at) AS first_snap, max(snapshot_created_at) AS last_snap
  FROM pos
  GROUP BY 1 ORDER BY snapshots DESC
""").df()

In [ ]:
# Gap between provider snapshot time and Decade ingest time — how 'late' are records?
con.execute("""
  SELECT institution_name,
         percentile_cont(0.5) WITHIN GROUP (ORDER BY epoch(cast(ingested_at as timestamp)) - epoch(cast(snapshot_created_at as timestamp)))/60 AS median_lag_min,
         percentile_cont(0.95) WITHIN GROUP (ORDER BY epoch(cast(ingested_at as timestamp)) - epoch(cast(snapshot_created_at as timestamp)))/60 AS p95_lag_min
  FROM pos GROUP BY 1 ORDER BY median_lag_min DESC
""").df()

### 2.4 · What — `investment_id`, `investment_type`, `payload_kind`

- **`investment_id`** — the provider's identifier for one holding. **This is the identifier the case study warns can churn.** The Open Finance spec (`variable-incomes/1.3.0.yml`) *mandates* reuse after 12-month dormancy, but institutions may not comply.
- **`investment_type`** — which product family. Maps 1:1 to the 5 sub-APIs of Open Finance Investments.
- **`payload_kind`** (positions only) — which endpoint the payload came from:
  - `balances` → `/investments/{investmentId}/balances` — the snapshot values (qty, market value)
  - `detail`   → `/investments/{investmentId}` — the security's static identity (ISIN, ticker, CNPJ, dueDate…)

**Modeling implication**: for each holding at each snapshot we receive **two records** (balance + detail). To get a single logical "position row" we must join them by `(institution_id, party_id, account_id, snapshot_id, investment_id)`. Detail can be considered slowly-changing (dueDate, ISIN don't move); balance changes every snapshot.

In [ ]:
con.execute("""
  SELECT investment_type, payload_kind, count(*) AS records,
         count(DISTINCT investment_id) AS holdings
  FROM pos GROUP BY 1,2 ORDER BY 1,2
""").df()

In [ ]:
# Verify: within a single snapshot, do we always get both balances and detail for each holding?
con.execute("""
  WITH per_kind AS (
    SELECT snapshot_id, investment_id, payload_kind, count(*) AS n
    FROM pos GROUP BY 1,2,3
  )
  SELECT b.n AS balances_per_snap, d.n AS details_per_snap, count(*) AS occurrences
  FROM (SELECT snapshot_id, investment_id, n FROM per_kind WHERE payload_kind='balances') b
  FULL JOIN (SELECT snapshot_id, investment_id, n FROM per_kind WHERE payload_kind='detail') d
    USING (snapshot_id, investment_id)
  GROUP BY 1,2 ORDER BY occurrences DESC
""").df()

### 2.5 · The odd columns — `snapshot_id`, `s3_uri`, `page`, `payload_source`

- **`snapshot_id`** — the grouping ID for one sync from one institution for one customer. A snapshot contains N holdings × 2 payloads.
- **`s3_uri`** — provenance: where in blob storage the raw JSON lives. Useful for audit; not for querying.
- **`page`** (transactions only) — Open Finance transactions endpoints paginate; this records which page this row came from.
- **`payload_source`** (transactions only) — which endpoint served the record: `transactions` (historical) or `transactions-current` (recent-only, ~7 days). Different completeness / freshness guarantees.

In [ ]:
con.execute("SELECT payload_source, count(*) FROM txn GROUP BY 1").df()

In [ ]:
# Snapshots — one per (institution, customer, sync run)
con.execute("""
  SELECT
    count(DISTINCT snapshot_id) AS snapshots,
    count(*) / count(DISTINCT snapshot_id)::float AS avg_records_per_snapshot
  FROM pos
""").df()

## 3 · Opening the JSON — position payloads

The payload structure differs per `(investment_type, payload_kind)`. Five families × two kinds = 10 payload shapes. Each cell below picks a real sample and shows it pretty-printed, with per-field notes from the OFB spec.

Helper to pull a sample:

In [ ]:
def sample_pos(inv_type: str, kind: str) -> dict:
    row = con.execute(
        "SELECT payload_json FROM pos WHERE investment_type=? AND payload_kind=? LIMIT 1",
        [inv_type, kind]
    ).fetchone()
    return json.loads(row[0])['data']  # unwrap the {data, meta, links} envelope

def sample_txn(inv_type: str) -> dict:
    row = con.execute(
        "SELECT transaction_json FROM txn WHERE investment_type=? LIMIT 1",
        [inv_type]
    ).fetchone()
    return json.loads(row[0])

### 3.1 · `VARIABLE_INCOMES` — equities, ETF, FII

**Spec**: [variable-incomes/1.3.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/variable-incomes/1.3.0.yml)

**detail** — the security's static identity. Very short: ISIN + ticker + issuer CNPJ. This is what makes variable-income the easiest family to derive a logical key for: `(isinCode, ticker)` is highly deterministic — those are B3-standardized.

**balances** — daily snapshot. `closingPrice` is the price on `referenceDate` (which is D-1 or D-2 per the spec).

In [ ]:
print('--- detail ---');   print(json.dumps(sample_pos('VARIABLE_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('VARIABLE_INCOMES','balances'), indent=2, ensure_ascii=False))

### 3.2 · `FUNDS` — investment funds

**Spec**: [funds/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/funds/1.1.0.yml)

**detail** — the fund's CNPJ is the natural logical key (every Brazilian fund has a unique CNPJ). ANBIMA fields are the industry taxonomy for categorizing the fund's strategy.

**balances** — funds are measured in *quotas*, not shares. `quotaGrossPriceValue` is the price of one quota on `referenceDate`. `grossAmount ≈ quotaQuantity × quotaGrossPriceValue` (identity check for us).

In [ ]:
print('--- detail ---');   print(json.dumps(sample_pos('FUNDS','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('FUNDS','balances'), indent=2, ensure_ascii=False))

### 3.3 · `BANK_FIXED_INCOMES` — CDB, LCI, LCA, LC, LF

**Spec**: [bank-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/bank-fixed-incomes/1.1.0.yml)

**detail** — richer than variable income. Includes `remuneration` (the interest formula: fixed, indexed to CDI/SELIC/IPCA), `dueDate`, `issueDate`, `clearingCode` (B3 identifier). These are *per-customer* securities (a CDB is issued to you specifically), so the logical key needs several fields: `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)` at minimum.

**balances** — includes `updatedUnitPrice` (marked-to-market unit value) and `purchaseUnitPrice` (what you paid). `netAmount` accounts for accrued income tax (`incomeTax`). Also carries `postFixedIndexerPercentage` — how much of the indexer you get (e.g. 102% of CDI).

In [ ]:
print('--- detail ---');   print(json.dumps(sample_pos('BANK_FIXED_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('BANK_FIXED_INCOMES','balances'), indent=2, ensure_ascii=False))

### 3.4 · `CREDIT_FIXED_INCOMES` — CRI, CRA, Debêntures

**Spec**: [credit-fixed-incomes/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/credit-fixed-incomes/1.1.0.yml)

Shape mirrors bank-fixed but adds `debtorCnpjNumber` / `debtorName` — the *ultimate* borrower (for a debênture, the company; for a CRI, the securitizer). Logical key candidate: `(isinCode, debtorCnpjNumber, dueDate)`.

**Tier-1 defect to watch**: sample below likely shows `issuerInstitutionCnpjNumber` ending in `.00` — that's the *tax-id decimal tail* from the case study, appearing in real synthetic data. This is what the raw contract will need to coerce.

In [ ]:
print('--- detail ---');   print(json.dumps(sample_pos('CREDIT_FIXED_INCOMES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('CREDIT_FIXED_INCOMES','balances'), indent=2, ensure_ascii=False))

### 3.5 · `TREASURE_TITLES` — Tesouro Direto (government bonds)

**Spec**: [treasure-titles/1.1.0.yml](https://github.com/OpenBanking-Brasil/openapi/blob/main/swagger-apis/treasure-titles/1.1.0.yml)

**detail** — standardized government product. `isinCode` alone is a strong logical key (every Tesouro title has a unique national ISIN). `productName` is human-readable ("Tesouro Prefixado 2027", "Tesouro IPCA+ 2035", ...).

**balances** — `updatedUnitPrice` reflects the bond's marked-to-market value (moves with rates). `purchaseUnitPrice` is what the customer paid. Difference × `quantity` ≈ unrealized gain.

In [ ]:
print('--- detail ---');   print(json.dumps(sample_pos('TREASURE_TITLES','detail'),   indent=2, ensure_ascii=False))
print('\n--- balances ---'); print(json.dumps(sample_pos('TREASURE_TITLES','balances'), indent=2, ensure_ascii=False))

## 4 · Opening the JSON — transaction payloads

Transactions are pre-flattened: one row per movement, not one row per API page. All five families share a broadly similar shape (`type`, `transactionType`, `transactionDate`, `transactionQuantity`, `transactionUnitPrice`, `transactionValue` / `transactionNetValue`, `transactionId`), with per-family additions.

Note the enums:
- **`type`** — direction: `ENTRADA` (money/quantity in) / `SAIDA` (out). Universal across families.
- **`transactionType`** — the semantic action: `COMPRA`, `APLICACAO`, `RESGATE`, `RENDIMENTO`, `JUROS`, `AMORTIZACAO`, ... Per-family enum in `EnumXxxTransactionsTransactionType`.

**Consumption layer implication**: the wealth page needs a *unified* type — user-facing categories like `buy`, `sell`, `income`, `transfer`. That mapping table lives in the consumption layer (or a shared contract), not in canonical.

In [ ]:
for it in ['VARIABLE_INCOMES','FUNDS','BANK_FIXED_INCOMES','CREDIT_FIXED_INCOMES','TREASURE_TITLES']:
    print(f'### {it} ###')
    print(json.dumps(sample_txn(it), indent=2, ensure_ascii=False))
    print()

In [ ]:
# What transactionType values actually appear per family?
con.execute("""
  SELECT investment_type,
         json_extract_string(transaction_json, '$.transactionType') AS tx_type,
         count(*) AS n
  FROM txn
  GROUP BY 1,2 ORDER BY 1, 3 DESC
""").df()

## 5 · A first pass at the defects

Now that we know the actual field names, wire up the three named defect probes. Keep results here — feeds directly into `design/data_quality.md`.

### 5.1 · Tier-1 · Tax-id decimal tail (from the case study, named)

Tax IDs (CNPJ, CPF) are 14 or 11 digit strings. Any value with `.` in it is the exact defect the case study warns about.

In [ ]:
con.execute("""
  SELECT investment_type,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.issuerInstitutionCnpjNumber') LIKE '%.%') AS decimal_tail_issuer,
         count(*) FILTER (WHERE json_extract_string(payload_json, '$.data.debtorCnpjNumber') LIKE '%.%')          AS decimal_tail_debtor,
         count(*) AS total
  FROM pos
  WHERE payload_kind = 'detail'
  GROUP BY 1 ORDER BY 1
""").df()

### 5.2 · Tier-2 · Intra-sync duplicate

For variable-income (easiest logical key), check: same `(snapshot_id, account_id, isinCode, ticker)` under >1 `investment_id`.

In [ ]:
con.execute("""
  WITH d AS (
    SELECT snapshot_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM pos
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT snapshot_id, account_id, isin, ticker,
         count(DISTINCT investment_id) AS n_ids,
         array_agg(DISTINCT investment_id) AS ids
  FROM d
  GROUP BY 1,2,3,4
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY n_ids DESC LIMIT 20
""").df()

### 5.3 · Tier-2 · Identity churn

Same logical key across snapshots under >1 `investment_id`. If any hits appear, the institution is violating the OFB spec's `investmentId` reuse mandate.

In [ ]:
con.execute("""
  WITH d AS (
    SELECT institution_id, account_id, investment_id,
           json_extract_string(payload_json, '$.data.isinCode') AS isin,
           json_extract_string(payload_json, '$.data.ticker')   AS ticker
    FROM pos
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
  )
  SELECT institution_id, account_id, isin, ticker,
         count(DISTINCT investment_id) AS n_ids,
         array_agg(DISTINCT investment_id) AS ids
  FROM d
  GROUP BY 1,2,3,4
  HAVING count(DISTINCT investment_id) > 1
  ORDER BY n_ids DESC LIMIT 20
""").df()

### 5.4 · Tier-2 · Zero-flap

Balance = 0 sandwiched between non-zero, quantity unchanged.

In [ ]:
con.execute("""
  WITH b AS (
    SELECT party_id, investment_id, snapshot_created_at,
           cast(json_extract_string(payload_json, '$.data.quantity') AS DOUBLE)                 AS qty,
           cast(json_extract_string(payload_json, '$.data.grossAmount.amount') AS DOUBLE)      AS gross
    FROM pos WHERE payload_kind = 'balances'
  ),
  o AS (
    SELECT *,
           lag(gross)  OVER w AS prev_v, lead(gross) OVER w AS next_v,
           lag(qty)    OVER w AS prev_q, lead(qty)   OVER w AS next_q
    FROM b
    WINDOW w AS (PARTITION BY party_id, investment_id ORDER BY snapshot_created_at)
  )
  SELECT party_id, investment_id, snapshot_created_at, qty, gross, prev_v, next_v
  FROM o
  WHERE gross = 0 AND prev_v > 0 AND next_v > 0 AND qty = prev_q AND qty = next_q
  ORDER BY snapshot_created_at LIMIT 20
""").df()

## 6 · Scratch-pad — observations for the design docs

Fill this in as you go. These bullets are what will become `design/decisions.md` and `design/data_quality.md`.

- **Grain of positions**: one snapshot × one holding = **two rows** (balances + detail). Canonical `positions` should be the join of the two, keyed on `(party_id, account_id, snapshot_id, investment_id)` at natural grain.
- **Watermark**: `ingested_at` on both files. `snapshot_created_at` is the event-time; the gap between them is the pipeline lag.
- **Logical key** per family — record what actually works when you probed above:
  - `VARIABLE_INCOMES`: `(isinCode, ticker)` — check that intra-sync dup query returned 0
  - `FUNDS`: `(cnpjNumber)` — write and run
  - `BANK_FIXED_INCOMES`: `(isinCode, issuerInstitutionCnpjNumber, dueDate, issueDate)` — write and run
  - `CREDIT_FIXED_INCOMES`: `(isinCode, debtorCnpjNumber, dueDate)` — write and run
  - `TREASURE_TITLES`: `(isinCode)` — write and run
- **Tier-1 defects observed**: fill in — decimal-tail count, missing-required count, enum violations.
- **Tier-2 defects observed**: fill in — intra-sync dup count per family, identity churn count per family, zero-flap count.
- **Transaction type mapping**: enumerate distinct values seen in the query above; group them into wealth-consumer categories (`buy`, `sell`, `income`, `transfer`, `tax`, `other`).